# Test PaddleOCR-VL-1.6 (qua OCR server nội bộ)

Gọi thẳng tới server OCR đã deploy (`OCR_BASE_URL` trong `.env`) qua API OpenAI-compatible, không chạy `paddleocr`/`paddlepaddle` local. Chỉ sửa `SOURCE_PATH`/`PAGE_NUMBER` rồi chạy tuần tự.

### Cấu hình
Đọc `OCR_ENGINE` / `OCR_MODEL` / `OCR_BASE_URL` / `OCR_API_KEY` từ `dev_llm_service/.env`. Cần `pymupdf` để render PDF ra ảnh — nếu chưa có, chạy 1 lần ở terminal (venv này do `uv` quản lý, không có `pip`):

```
uv add --group dev pymupdf
```

In [3]:
import base64
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path("../.env"))

print("Kernel Python:", sys.executable)
print("OCR_ENGINE   :", os.getenv("OCR_ENGINE"))
print("OCR_MODEL    :", os.getenv("OCR_MODEL"))
print("OCR_BASE_URL :", os.getenv("OCR_BASE_URL"))

Kernel Python: c:\Users\khanh\AppData\Local\Programs\Python\Python313\python.exe
OCR_ENGINE   : paddleocr
OCR_MODEL    : paddle-ocr-v1.6
OCR_BASE_URL : http://172.20.0.59:8081


In [4]:
from openai import OpenAI

ocr_client = OpenAI(
    base_url=os.getenv("OCR_BASE_URL").rstrip("/") + "/v1",
    api_key=os.getenv("OCR_API_KEY", "EMPTY"),
)
OCR_MODEL = os.getenv("OCR_MODEL")

In [1]:
!pip install PyMuPDF


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import fitz  # PyMuPDF

### Cấu hình input
`SOURCE_PATH` có thể là ảnh (`.png/.jpg`) hoặc PDF. Nếu là PDF thì cell render bên dưới sẽ cắt `PAGE_NUMBER` ra thành ảnh trước khi OCR.

In [5]:
SOURCE_PATH = Path("../my_documents/Bai_Tap_SQL_to_Text_Agent.pdf")
PAGE_NUMBER = 1  # chỉ dùng nếu SOURCE_PATH là PDF
OUTPUT_DIR = Path("output_paddleocr_vl")
OUTPUT_DIR.mkdir(exist_ok=True)

if not SOURCE_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy file: {SOURCE_PATH}")

print("Input file:", SOURCE_PATH)

Input file: ..\my_documents\Bai_Tap_SQL_to_Text_Agent.pdf


In [6]:
def resolve_image_path(source_path: Path, page_number: int) -> Path:
    """Trả về đường dẫn ảnh để OCR. Nếu source là PDF thì render trang ra PNG."""
    if source_path.suffix.lower() != ".pdf":
        return source_path

    doc = fitz.open(str(source_path))
    page = doc[page_number - 1]
    pix = page.get_pixmap(dpi=400, alpha=False)

    image_path = OUTPUT_DIR / f"{source_path.stem}_page_{page_number}.png"
    pix.save(str(image_path))
    doc.close()
    return image_path


image_path = resolve_image_path(SOURCE_PATH, PAGE_NUMBER)
print("Ảnh dùng để OCR:", image_path)

Ảnh dùng để OCR: output_paddleocr_vl\Bai_Tap_SQL_to_Text_Agent_page_1.png


### Gọi OCR server và xem kết quả
Server serve PaddleOCR-VL qua API dạng OpenAI-compatible (vision chat completion). Prompt `"OCR:"` là convention chính thức của PaddleOCR-VL để lấy toàn bộ text theo thứ tự đọc — nếu server bạn cấu hình endpoint/response khác, chỉnh lại cell dưới.

In [ ]:
def ocr_image(image_path: Path, prompt: str = "OCR:") -> str:
    b64 = base64.b64encode(image_path.read_bytes()).decode()
    response = ocr_client.chat.completions.create(
        model=OCR_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{b64}"},
                    },
                    {"type": "text", "text": prompt},
                ],
            }
        ],
        max_tokens=4096,
        temperature=0,
    )
    return response.choices[0].message.content

In [ ]:
start = time.perf_counter()
text = ocr_image(image_path)
elapsed = time.perf_counter() - start

print("=" * 100)
print(f"PaddleOCR-VL ({OCR_MODEL}) time: {elapsed:.2f}s")
print("=" * 100)
print(text)

out_file = OUTPUT_DIR / f"{image_path.stem}.md"
out_file.write_text(text, encoding="utf-8")
print(f"\nĐã lưu kết quả vào: {out_file.resolve()}")